In [0]:
%sql
CREATE OR REPLACE TABLE medical_pipeline.gold.datacube AS
WITH encounter_base AS (
  SELECT 
    d.year,
    d.quarter,
    d.month,
    d.half_year,
    fe.patient_id,
    fe.payer_id,
    fe.encounter_id,
    fe.encounter_class,
    fe.total_cost AS total_claim_cost,
    fe.payer_coverage,
    fe.has_payer_coverage,
    DATEDIFF(fe.stop, fe.start) AS encounter_duration_days
  FROM medical_pipeline.gold.fact_encounters fe
  LEFT JOIN medical_pipeline.gold.dim_dates d ON DATE(fe.start) = d.date
),
procedure_agg AS (
  SELECT 
    d.year,
    d.quarter,
    d.month,
    fp.encounter_id,
    COUNT(*) AS procedure_count,
    SUM(fp.base_cost) AS total_procedure_cost
  FROM medical_pipeline.gold.fact_procedures fp
  LEFT JOIN medical_pipeline.gold.dim_dates d ON fp.procedure_date = d.date
  GROUP BY d.year, d.quarter, d.month, fp.encounter_id
)
SELECT 
  eb.year,
  eb.quarter,
  eb.month,
  eb.half_year,
  eb.payer_id,
  eb.encounter_class,
  COUNT(DISTINCT eb.encounter_id) AS total_encounters,
  COUNT(DISTINCT eb.patient_id) AS unique_patients,
  SUM(eb.total_claim_cost) AS total_encounter_cost,
  SUM(eb.payer_coverage) AS total_payer_coverage,
  SUM(COALESCE(pa.procedure_count, 0)) AS total_procedures,
  SUM(COALESCE(pa.total_procedure_cost, 0)) AS total_procedure_cost
FROM encounter_base eb
LEFT JOIN procedure_agg pa 
  ON eb.encounter_id = pa.encounter_id 
  AND eb.year = pa.year 
  AND eb.quarter = pa.quarter 
  AND eb.month = pa.month
GROUP BY 
  eb.year, eb.quarter, eb.month, eb.half_year,
  eb.payer_id, eb.encounter_class
ORDER BY 
  eb.year, eb.quarter, eb.month, eb.encounter_class;

SELECT 'Master Unified Table Created' AS status, COUNT(*) AS total_rows 
FROM medical_pipeline.gold.datacube

In [0]:
# from pyspark.sql.functions import col

fact_df = spark.table("medical_pipeline.gold.fact_encounters")
date_df = spark.table("medical_pipeline.gold.dim_dates")
     

from pyspark.sql.functions import to_date, quarter, when

df = fact_df.withColumn(
    "encounter_date",
    to_date(col("start"))
).join(
    date_df,
    col("encounter_date") == col("date"),
    "left"
)

df = df.withColumn(
    "quarter",
    quarter(col("encounter_date"))
)

df = df.withColumn("year", col("year").cast("int")) \
       .withColumn("month", col("month").cast("int"))

df = df.withColumn(
    "duration_category",
    when(col("more_than_24") == 1, "More than 24 hours")
    .otherwise("24 hours or less")
)
     

from pyspark.sql.functions import count, sum, col, coalesce, lit

cube_for_save = df.cube(
    "year",
    "month",
    "quarter",
    "payer_id",
    "encounter_class",
    "duration_category"
).agg(
    count("encounter_id").alias("encounter_count"),
    sum("total_cost").alias("total_cost")
)

cube_for_save = cube_for_save \
    .withColumn("year", coalesce(col("year").cast("string"), lit("ALL"))) \
    .withColumn("month", coalesce(col("month").cast("string"), lit("ALL"))) \
    .withColumn("quarter", coalesce(col("quarter").cast("string"), lit("ALL"))) \
    .withColumn("payer_id", coalesce(col("payer_id"), lit("ALL"))) \
    .withColumn("encounter_class", coalesce(col("encounter_class"), lit("ALL"))) \
    .withColumn("duration_category", coalesce(col("duration_category"), lit("ALL")))

cube_for_save.write.format("delta") \
    .mode("overwrite") \
    .saveAsTable("medical_pipeline.gold.encounter_cube")

In [0]:
%sql
select * from medical_pipeline.gold.datacube
